# Fusion expérimentale des trois modules

Ce notebook définit et teste le score global utilisé dans l'application Streamlit.

- 35 % risque microplastique
- 30 % risque HAB
- 35 % risque visuel

Les seuils sont : `Healthy < 0,33`, `At risk < 0,66`, sinon `Degraded`.

In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()
if (cwd / 'app').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'app').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Le dossier app est introuvable.")

import sys
sys.path.insert(0, str(PROJECT_ROOT / 'app'))

from fusion_engine import compute_global_risk

print('Racine du projet :', PROJECT_ROOT)

Racine du projet : C:\Users\elasr\OneDrive\Desktop\hind


In [2]:
examples = pd.DataFrame([
    {'scenario': 'Faible risque', 'microplastic': 0.15, 'hab': 0.10, 'visual': 0.20},
    {'scenario': 'Risque intermédiaire', 'microplastic': 0.55, 'hab': 0.40, 'visual': 0.50},
    {'scenario': 'Risque élevé', 'microplastic': 0.85, 'hab': 0.90, 'visual': 0.75},
])

results = []
for _, row in examples.iterrows():
    fusion = compute_global_risk(
        row['microplastic'], row['hab'], row['visual']
    )
    results.append({
        **row.to_dict(),
        'global_risk': fusion.global_risk,
        'status': fusion.status,
    })

results_df = pd.DataFrame(results)
display(results_df)

results_dir = PROJECT_ROOT / 'results' / 'fusion'
results_dir.mkdir(parents=True, exist_ok=True)
results_df.to_csv(results_dir / 'fusion_examples.csv', index=False, encoding='utf-8-sig')

,scenario,microplastic,hab,visual,global_risk,status
0,Faible risque,0.15,0.1,0.20,0.1525,Healthy
1,Risque intermédiaire,0.55,0.4,0.50,0.4875,At risk
2,Risque élevé,0.85,0.9,0.75,0.8300,Degraded


In [3]:
metadata = {
    'method': 'experimental_rule_based_fusion',
    'weights': {
        'microplastic': 0.35,
        'hab': 0.30,
        'visual': 0.35,
    },
    'thresholds': {
        'Healthy': '[0.00, 0.33)',
        'At risk': '[0.33, 0.66)',
        'Degraded': '[0.66, 1.00]',
    },
    'warning': 'Not a field-validated ecological diagnosis.',
}

models_dir = PROJECT_ROOT / 'models' / 'fusion'
models_dir.mkdir(parents=True, exist_ok=True)
(models_dir / 'fusion_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2),
    encoding='utf-8'
)

print('✅ Métadonnées de fusion sauvegardées.')

✅ Métadonnées de fusion sauvegardées.
